In [1]:
import os
import time
from pathlib import Path

import whisper
import torch

from vosk import Model, KaldiRecognizer

import soundfile as sf
import json

In [2]:
print("="*50)
print("Información del entorno")
print("="*50)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Procesamiento en CPU")

Información del entorno
PyTorch: 2.12.1+cpu
CUDA disponible: False
Procesamiento en CPU


In [3]:
#Rutas
ROOT = Path("../")

AUDIO_DIR = ROOT / "data" / "processed"

MODELS_DIR = ROOT / "models"

REFERENCE_DIR = ROOT / "data" / "references"

In [4]:
#Cargar modelo Whisper
print("Cargando Whisper Base...")

device = "cuda" if torch.cuda.is_available() else "cpu"

whisper_model = whisper.load_model(
    "base",
    device=device
)

print("Modelo Whisper cargado correctamente.")

Cargando Whisper Base...


100%|███████████████████████████████████████| 139M/139M [00:30<00:00, 4.83MiB/s]


Modelo Whisper cargado correctamente.


In [5]:
#Importar Vosk
from vosk import Model

vosk_path = MODELS_DIR / "vosk-model-es-0.42"

vosk_model = Model(str(vosk_path))

print("Modelo Vosk cargado correctamente.")

Modelo Vosk cargado correctamente.


In [6]:
#Seleccionar un archivo de audio para prueba
audio_prueba = sorted(AUDIO_DIR.glob("*.wav"))[0]

print(audio_prueba.name)

spontaneous-speech-es-71834.wav


In [8]:
#Validar Whisper
import librosa

inicio = time.perf_counter()

audio, sr = librosa.load(
    audio_prueba,
    sr=16000,
    mono=True
)

resultado = whisper_model.transcribe(
    audio,
    language="es",
    fp16=False
)

fin = time.perf_counter()

print("="*50)
print("Whisper")
print("="*50)

print(resultado["text"])

print()

print(f"Tiempo: {fin-inicio:.3f} segundos")

d:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Whisper
 My festividad favorita es la Navidad, se festeja en familia con mucha comida y muchos reales.

Tiempo: 41.546 segundos


In [9]:
#Validar Vosk
data, samplerate = sf.read(audio_prueba)

if len(data.shape) > 1:
    data = data[:,0]

rec = KaldiRecognizer(vosk_model, samplerate)

rec.AcceptWaveform(
    (data * 32767).astype("int16").tobytes()
)

resultado = json.loads(rec.FinalResult())

print("="*50)
print("Vosk")
print("="*50)

print(resultado["text"])

Vosk
y festividad favorita es la navidad se festeja en familia con mucha comida y muchos regalos


In [14]:
#Resumen
print("="*40)
print("VALIDACIÓN COMPLETADA")
print("="*40)

print("Whisper Base      OK")

print("Vosk              OK")

print("Entorno listo para procesamiento")

VALIDACIÓN COMPLETADA
Whisper Base      OK
Vosk              OK
Entorno listo para procesamiento
